# 04 — Ghép fact và huấn luyện Spark LR/GBT

### 1. Dựng fact từ Silver và dimension

Chạy sau notebook 03. `build_fact()` ghép mỗi đơn Silver với phiên bản
shipper theo ngày nhận và các khóa vùng, AOI, ngày nhận, ngày giao, giờ nhận.
Hàm ghi `fact_delivery_task_gold` và đối soát khóa với Silver.

In [ ]:
import os

from datetime import datetime

import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor, LinearRegression
from pyspark.sql.window import Window

from lakehouse.common import attach_dimension_keys, join_courier_history
from lakehouse.notebooks import load_notebook_steps

shared = load_notebook_steps()
DIM = shared.DIM
DIMS = shared.DIMS
FACT = shared.FACT
NUM_FEATURES = shared.NUM_FEATURES
PRED = shared.PRED
RUN_INFO = shared.RUN_INFO
SILVER = shared.SILVER
reconcile_order_keys = shared.reconcile_order_keys
write_mysql = shared.write_mysql

spark = shared.get_spark("Delivery - Fact and ML Gold")

In [ ]:
def build_fact(spark):
    silver = spark.read.format("delta").load(SILVER)
    dim = spark.read.format("delta").load(DIM)
    fact = join_courier_history(silver, dim)
    fact = attach_dimension_keys(fact, {n: spark.read.format("delta").load(p) for n, p in DIMS.items()})
    fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(FACT)
    fact = spark.read.format("delta").load(FACT)
    reconcile_order_keys(silver, fact, "Silver → Fact")
    return fact

In [ ]:
fact = build_fact(spark)

In [2]:
fact.select(
    "order_id", "courier_sk", "region_sk", "aoi_sk", "accept_date_key",
    "delivery_date_key", "accept_hour_key", "delivery_minutes",
).show(5, truncate=False)
print("Số dòng fact:", fact.count())
print("order_id duy nhất:", fact.select("order_id").distinct().count())

+--------+--------------------+---------+------+---------------+-----------------+---------------+----------------+
|order_id|courier_sk          |region_sk|aoi_sk|accept_date_key|delivery_date_key|accept_hour_key|delivery_minutes|
+--------+--------------------+---------+------+---------------+-----------------+---------------+----------------+
|2071340 |-5931354023872468343|1        |14    |20220618       |20220618         |8              |501.0           |
|3767291 |-5931354023872468343|1        |14    |20220814       |20220814         |8              |391.0           |
|2184812 |-5931354023872468343|1        |14    |20220901       |20220901         |9              |367.0           |
|4048066 |-6655371896403481998|1        |28    |20220825       |20220825         |13             |267.0           |
|2309133 |-306575199876071777 |1        |29    |20220709       |20220709         |8              |119.0           |
+--------+--------------------+---------+------+---------------+--------

### 2. Xem tập dùng cho Spark

Giữ đơn có GPS lúc nhận hợp lệ, khoảng cách tới đích tối đa 50 km và thời
lượng tối đa 24 giờ. Cell gọi đúng bộ lọc của Spark LR/GBT.

In [ ]:
def spark_baseline_cohort(fact):
    """Select the GPS, distance and duration cohort used by Spark LR/GBT."""
    return fact.filter(F.col("eligible_ml_distance") & (F.col("delivery_minutes") <= 1440)
                       & (F.col("distance_km") <= 50))

In [ ]:
cohort = spark_baseline_cohort(fact)
print("Số đơn đủ điều kiện vào tập học Spark", cohort.count())
cohort.select("order_id", "order_date", "delivery_minutes", "distance_km").show(5, truncate=False)

### 3. Chia theo ngày và fit hai baseline

Train gồm ngày nhận trước 01/10/2022. Test bắt đầu từ ngày đó. Hai mô hình
học `log1p(delivery_minutes)`. Khi chấm thì đổi bằng `expm1` về phút và chặn dưới ở 0.

`train_spark_baseline()` ghi `eta_predictions_gold`. Prefect gọi
`build_fact_and_train()` để dựng fact và fit liền nhau.

`_train_baseline()` tạo đặc trưng trong ngày theo shipper, mã hóa biến
vùng và AOI, fit LR/GBT rồi chấm trên test. `train_spark_baseline()`
quản lý các bảng cache trong lúc fit. `build_fact_and_train()` là lời gọi
gộp hai phần cho Prefect.

In [ ]:
def _metrics(df, label, pred_col):
    r = df.select(
        F.count("*").alias("n"),
        F.avg(F.abs(F.col(pred_col) - F.col(label))).alias("mae"),
        F.sqrt(F.avg((F.col(pred_col) - F.col(label)) ** 2)).alias("rmse"),
        F.sum((F.col(pred_col) - F.col(label)) ** 2).alias("sse"),
        F.sum((F.col(label) - F.lit(df.agg(F.avg(label)).first()[0])) ** 2).alias("sst"),
    ).first()
    r2 = float(1 - r["sse"] / r["sst"]) if r["sst"] else None
    return r["n"], float(r["mae"]), float(r["rmse"]), r2

In [ ]:
def _train_baseline(fact, train_end, late_threshold_min, run_id, cached):
    spark = fact.sparkSession
    n_silver = fact.count()

    ml = spark_baseline_cohort(fact)
    w = (Window.partitionBy("courier_id", "order_date").orderBy("accept_ts", "order_id")
         .rowsBetween(Window.unboundedPreceding, Window.currentRow))
    ml = (ml.withColumn("accept_hour", F.hour("accept_ts") + F.minute("accept_ts") / 60)
            .withColumn("dow", F.dayofweek("order_date"))
            .withColumn("order_rank", F.count("*").over(w).cast("double"))
            .withColumn("cum_avg_dist", F.avg("distance_km").over(w))
            .withColumn("aoi_type_str", F.col("aoi_type").cast("string"))
            .withColumn("region_str", F.col("region_id").cast("string"))
            .withColumn("label_log", F.log1p("delivery_minutes")))

    train = ml.filter(F.col("order_date") < train_end)
    test = ml.filter(F.col("order_date") >= train_end)

    # Lịch sử hiệu suất shipper: chỉ tính từ tập train
    cstat = train.groupBy("courier_id").agg(F.avg("delivery_minutes").alias("courier_train_avg_min"))
    global_avg = train.agg(F.avg("delivery_minutes")).first()[0]
    if global_avg is None:
        raise ValueError("Tập train rỗng, kiểm tra train_end")
    train = train.join(cstat, "courier_id", "left").fillna({"courier_train_avg_min": global_avg})
    test = test.join(cstat, "courier_id", "left").fillna({"courier_train_avg_min": global_avg})
    train, test = train.cache(), test.cache()
    cached.extend([train, test])
    n_train, n_test = train.count(), test.count()
    if n_train < 2 or n_test < 2:
        raise ValueError("Train/test quá nhỏ, chỉnh train_end")

    idx = Pipeline(stages=[
        StringIndexer(inputCol="aoi_type_str", outputCol="aoi_idx", handleInvalid="keep"),
        StringIndexer(inputCol="region_str", outputCol="region_idx", handleInvalid="keep"),
    ]).fit(train)
    tr, te = idx.transform(train), idx.transform(test)
    ohe = OneHotEncoder(inputCols=["aoi_idx", "region_idx"], outputCols=["aoi_ohe", "region_ohe"],
                        handleInvalid="keep").fit(tr)
    tr, te = ohe.transform(tr), ohe.transform(te)
    a_lr = VectorAssembler(inputCols=NUM_FEATURES + ["aoi_ohe", "region_ohe"], outputCol="features_lr")
    a_gbt = VectorAssembler(inputCols=NUM_FEATURES + ["aoi_idx", "region_idx"], outputCol="features_gbt")
    tr = a_gbt.transform(a_lr.transform(tr))
    te = a_gbt.transform(a_lr.transform(te))

    lr = LinearRegression(featuresCol="features_lr", labelCol="label_log",
                          predictionCol="p_lr_log").fit(tr)
    gbt = GBTRegressor(featuresCol="features_gbt", labelCol="label_log", predictionCol="p_gbt_log",
                       maxIter=80, maxDepth=6, maxBins=256, seed=42).fit(tr)

    med = train.approxQuantile("delivery_minutes", [0.5], 0.01)[0]
    pred = (gbt.transform(lr.transform(te))
            .withColumn("pred_lr", F.greatest(F.expm1("p_lr_log"), F.lit(0.0)))
            .withColumn("pred_gbt", F.greatest(F.expm1("p_gbt_log"), F.lit(0.0)))
            .withColumn("pred_median", F.lit(float(med)))
            .withColumn("pred_mean", F.lit(float(global_avg)))
            .select("order_id", "courier_id", "region_id", "aoi_type", "order_date", "distance_km",
                    "accept_hour", "delivery_minutes", "pred_lr", "pred_gbt", "pred_median", "pred_mean")
            .cache())
    cached.append(pred)
    pred.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(PRED)

    run_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    rows = []
    for name, col in [("Baseline: trung vị", "pred_median"), ("Baseline: trung bình", "pred_mean"),
                      ("Linear Regression", "pred_lr"), ("GBT", "pred_gbt")]:
        n, mae, rmse, r2 = _metrics(pred, "delivery_minutes", col)
        rows.append((name, "test (<=24h, GPS hợp lệ)", n, round(mae, 2), round(rmse, 2),
                     round(r2, 4) if r2 is not None else None, run_ts))
    b_mae, b_rmse = os.getenv("BENCHMARK_MAE"), os.getenv("BENCHMARK_RMSE")
    if b_mae and b_rmse:
        rows.append((os.getenv("BENCHMARK_NAME", "Benchmark bài báo gốc"), "theo bài báo",
                     None, float(b_mae), float(b_rmse), None, run_ts))
    metrics = spark.createDataFrame(
        rows, "model string, cohort string, n long, mae double, rmse double, r2 double, run_ts string")
    write_mysql(metrics, "ml_metrics", run_id=run_id)

    names = NUM_FEATURES + ["aoi_type", "region_id"]
    imp = spark.createDataFrame(
        [(n_, float(v)) for n_, v in zip(names, gbt.featureImportances.toArray())],
        "feature string, importance double")
    write_mysql(imp, "ml_feature_importance", run_id=run_id)

    if late_threshold_min is None:
        late_threshold_min = float(train.approxQuantile("delivery_minutes", [0.9], 0.01)[0])
    info = spark.createDataFrame(
        [(run_ts, str(train_end), n_silver, n_train, n_test, float(late_threshold_min))],
        "run_ts string, train_end string, silver_rows long, train_rows long, test_rows long, "
        "late_threshold_min double")
    info.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(RUN_INFO)
    write_mysql(info, "pipeline_run_info", run_id=run_id)
    out = {r[0]: {"mae": r[3], "rmse": r[4], "r2": r[5]} for r in rows}
    out.update({"train_rows": n_train, "test_rows": n_test, "late_threshold_min": late_threshold_min})
    return out

In [ ]:
def train_spark_baseline(fact, train_end="2022-10-01", late_threshold_min=None, *, run_id=None):
    cached = []
    try:
        return _train_baseline(fact, train_end, late_threshold_min, run_id, cached)
    finally:
        for frame in reversed(cached):
            frame.unpersist()


def build_fact_and_train(spark, train_end="2022-10-01", late_threshold_min=None, *, run_id=None):
    return train_spark_baseline(build_fact(spark), train_end, late_threshold_min, run_id=run_id)

In [1]:
result = train_spark_baseline(fact, train_end="2022-10-01")
result

{'Baseline: trung vị': {'mae': 127.59, 'rmse': 169.45, 'r2': -0.0494},
 'Baseline: trung bình': {'mae': 131.0, 'rmse': 165.42, 'r2': -0.0001},
 'Linear Regression': {'mae': 109.88, 'rmse': 168.64, 'r2': -0.0394},
 'GBT': {'mae': 92.29, 'rmse': 136.19, 'r2': 0.322},
 'train_rows': 165222,
 'test_rows': 35778,
 'late_threshold_min': 467.0}